In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost as xgb

# Load dữ liệu vào biến df
df = pd.read_csv('../data/raw/full_data.csv')
print("Kích thước dữ liệu:", df.shape)
# Kiểm tra xem có cột nào bị thiếu dữ liệu không (sẽ in ra 0 hết)
print("Số lượng dữ liệu thiếu:\n", df.isnull().sum().sum())
df.head()


# BƯỚC 1: TIỀN XỬ LÝ (MÃ HÓA CƠ BẢN)

def encode_data(data):
    df_encoded = data.copy()
    
    # 1.1 Label Encoding (Biến nhị phân)
    binary_cols = ['gender', 'ever_married', 'Residence_type']
    for col in binary_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])
        
    # 1.2 One-hot Encoding (Biến đa lớp)
    multi_cols = ['work_type', 'smoking_status']
    df_encoded = pd.get_dummies(df_encoded, columns=multi_cols, drop_first=True)
    
    return df_encoded

df_processed = encode_data(df)

# Tách Feature (X) và Target (y)
X = df_processed.drop('stroke', axis=1)
y = df_processed['stroke']


# BƯỚC 2: CÔ LẬP TẬP TEST (TRAIN/TEST SPLIT)

# Phải tách dữ liệu TRƯỚC KHI dùng SMOTE và Scaler
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# BƯỚC 3: CHUẨN HÓA (SCALING) TRÁNH RÒ RỈ

numerical_cols = ['age', 'avg_glucose_level', 'bmi']
scaler = StandardScaler()

# CHÚ Ý: Chỉ 'fit' (học) trên tập Train, sau đó 'transform' cho cả Train và Test
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


# BƯỚC 4: ISOLATED SMOTE (CÂN BẰNG TẬP TRAIN)

print(f"Trước SMOTE - Tập Train: 0={sum(y_train==0)}, 1={sum(y_train==1)}")

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Sau SMOTE - Tập Train: 0={sum(y_train_smote==0)}, 1={sum(y_train_smote==1)}")
print(f"Tập Test GIỮ NGUYÊN: 0={sum(y_test==0)}, 1={sum(y_test==1)}")


# BƯỚC 5: HUẤN LUYỆN VÀ CHẤM ĐIỂM

# Khởi tạo mô hình XGBoost
model = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

# Huấn luyện trên tập đã được SMOTE
model.fit(X_train_smote, y_train_smote)

# Dự đoán trên tập Test (Tập tự nhiên, chưa từng nhìn thấy)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1] # Lấy xác suất % mắc bệnh

# Chấm điểm
print("\n" + "="*40)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH XGBOOST")
print("="*40)
print(f"Accuracy (Độ chính xác): {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (Độ chuẩn xác): {precision_score(y_test, y_pred):.4f}")
print(f"Recall (Độ nhạy - Quan trọng nhất): {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Kích thước dữ liệu: (4981, 11)
Số lượng dữ liệu thiếu:
 0
Trước SMOTE - Tập Train: 0=3786, 1=198
Sau SMOTE - Tập Train: 0=3786, 1=3786
Tập Test GIỮ NGUYÊN: 0=947, 1=50

KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH XGBOOST
Accuracy (Độ chính xác): 0.8927
Precision (Độ chuẩn xác): 0.1392
Recall (Độ nhạy - Quan trọng nhất): 0.2200
F1-Score: 0.1705
ROC-AUC: 0.7883

Confusion Matrix:
[[879  68]
 [ 39  11]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.93      0.94       947
           1       0.14      0.22      0.17        50

    accuracy                           0.89       997
   macro avg       0.55      0.57      0.56       997
weighted avg       0.92      0.89      0.90       997



In [ ]:
import lightgbm as lgb
import time
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print("🚀 Bắt đầu huấn luyện LightGBM...")
thoi_gian_bat_dau = time.time()

# 1. Khởi tạo "Bộ não" LightGBM
# Các tham số này đã được tinh chỉnh cơ bản cho bài toán phân loại nhị phân
lgbm_model = lgb.LGBMClassifier(
    n_estimators=200,        # Đọc 200 cuốn sách quy luật (Giống XGBoost)
    learning_rate=0.1,       # Tốc độ học
    num_leaves=31,           # ĐẶC SẢN CỦA LIGHTGBM: Số lượng lá tối đa trên mỗi cây (Mặc định 31 là rất tốt)
    max_depth=-1,            # Không giới hạn độ sâu (Để num_leaves tự kiểm soát)
    class_weight='balanced', # Tự động phạt nặng nếu AI đoán sai người bệnh (Hỗ trợ thêm cho SMOTE)
    random_state=42,
    n_jobs=-1                # Huy động 100% sức mạnh CPU của máy tính để chạy nhanh nhất
)

# 2. Cho AI đi học
lgbm_model.fit(X_train, y_train)

thoi_gian_ket_thuc = time.time()
print(f"✅ Huấn luyện hoàn tất! Thời gian chạy: {thoi_gian_ket_thuc - thoi_gian_bat_dau:.2f} giây")



# 1. Yêu cầu AI dự đoán trên tập dữ liệu kiểm tra (X_test) lightgbm
y_pred = lgbm_model.predict(X_test)               # AI chốt hạ: 0 (Khỏe) hoặc 1 (Bệnh)
y_pred_proba = lgbm_model.predict_proba(X_test)[:, 1] # Xác suất phần trăm rủi ro

# 2. Bóc tách Ma trận nhầm lẫn để tính Specificity
# tn: True Negative (Đoán khỏe đúng), fp: False Positive (Báo động giả)
# fn: False Negative (Bỏ lọt bệnh), tp: True Positive (Đoán bệnh đúng)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# 3. Tính toán 6 chỉ số y hệt như trong ảnh
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred) # Recall chính là Sensitivity
specificity = tn / (tn + fp)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

# 4. In ra bảng kết quả cực đẹp để copy vào Word/Excel
ket_qua = pd.DataFrame({
    'Metric': ['Accuracy', 'AUC', 'F1-Score', 'Sensitivity', 'Specificity', 'Precision'],
    'Điểm số': [f"{accuracy*100:.2f}%", f"{auc*100:.2f}%", f"{f1*100:.2f}%", 
                f"{sensitivity*100:.2f}%", f"{specificity*100:.2f}%", f"{precision*100:.2f}%"]
})

print("📊 BẢNG ĐÁNH GIÁ NĂNG LỰC AI DỰ ĐOÁN ĐỘT QUỴ")
print("-" * 45)
print(ket_qua.to_string(index=False))


In [ ]:
import time
from catboost import CatBoostClassifier
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print("🐅 Bắt đầu huấn luyện CatBoost...")
thoi_gian_bat_dau = time.time()

# 1. Khởi tạo não bộ CatBoost
cat_model = CatBoostClassifier(
    iterations=200,             # Đọc 200 cuốn sách quy luật
    learning_rate=0.1,          # Tốc độ học
    depth=6,                    # Độ sâu của cây quyết định
    auto_class_weights='Balanced', # TÍNH NĂNG XỊN: Tự động phạt lỗi nặng nếu đoán sai người bệnh (Cân bằng y tế)
    random_seed=42,
    verbose=0                   # Tắt các dòng log chạy rào rào cho màn hình đỡ rối
)

# 2. Bắt đầu học (Dùng luôn tập X_train, y_train từ đầu)
cat_model.fit(X_train, y_train)

thoi_gian_ket_thuc = time.time()
print(f"✅ Huấn luyện CatBoost hoàn tất! Thời gian: {thoi_gian_ket_thuc - thoi_gian_bat_dau:.2f} giây")



# 1. Bắt CatBoost làm bài thi trên đề thi X_test
y_pred_cat = cat_model.predict(X_test)
y_pred_proba_cat = cat_model.predict_proba(X_test)[:, 1]

# 2. Tính toán ma trận nhầm lẫn
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_cat).ravel()

# 3. Tính 6 chỉ số y khoa
accuracy = accuracy_score(y_test, y_pred_cat)
precision = precision_score(y_test, y_pred_cat)
sensitivity = recall_score(y_test, y_pred_cat)
specificity = tn / (tn + fp)
f1 = f1_score(y_test, y_pred_cat)
auc = roc_auc_score(y_test, y_pred_proba_cat)

# 4. In bảng điểm
ket_qua_cat = pd.DataFrame({
    'Metric': ['Accuracy', 'AUC', 'F1-Score', 'Sensitivity', 'Specificity', 'Precision'],
    'Điểm CatBoost': [f"{accuracy*100:.2f}%", f"{auc*100:.2f}%", f"{f1*100:.2f}%", 
                      f"{sensitivity*100:.2f}%", f"{specificity*100:.2f}%", f"{precision*100:.2f}%"]
})

print("📊 BẢNG ĐÁNH GIÁ NĂNG LỰC CATBOOST")
print("-" * 40)
print(ket_qua_cat.to_string(index=False))

In [ ]:
import joblib
import os

print("📦 Đang tiến hành đóng gói các mô hình AI...")



# 2. Lưu mô hình XGBoost (Biến của bạn tên là 'model')
joblib.dump(model, '../trained_models/xgboost_cdc_stroke_model.pkl')
print("✅ Đã lưu thành công: XGBoost")

# 3. Lưu mô hình LightGBM (Biến của bạn tên là 'lgbm_model')
# (Nếu bạn chưa chạy thành công LightGBM thì có thể thêm dấu # ở trước dòng này để ẩn nó đi)
joblib.dump(lgbm_model, '../trained_models/lightgbm_cdc_stroke_model.pkl')
print("✅ Đã lưu thành công: LightGBM")

# 4. Lưu mô hình CatBoost (Biến của bạn tên là 'cat_model')
# (Nếu bạn chưa chạy thành công CatBoost thì có thể thêm dấu # ở trước dòng này để ẩn nó đi)
joblib.dump(cat_model, '../trained_models/catboost_cdc_stroke_model.pkl')
print("✅ Đã lưu thành công: CatBoost")

print("🎉 Hoàn tất! Hãy kiểm tra thư mục 'trained_models' bên trái màn hình nhé.")